In [1]:
import pandas as pd
import numpy as np
import jenkspy

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

from pulp import (
    LpProblem,
    LpVariable,
    LpMaximize,
    lpSum,
    PULP_CBC_CMD,
    value,
)

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"
OUTPUT_CSV = Path("data") / "vulnerability.csv"

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# DEA CRS MULTIPLIER MODEL
# =============================================================================

def dea_crs(dmu_df, input_cols, output_cols):

    dmus = dmu_df.index.tolist()

    efficiencies = []

    for dmu in dmus:

        prob = LpProblem(
            f"DEA_{dmu}",
            LpMaximize
        )

        # ---------------------------------------------------------------------
        # OUTPUT WEIGHTS
        # ---------------------------------------------------------------------

        u = {
            col: LpVariable(
                f"u_{col}_{dmu}",
                lowBound=1e-6
            )
            for col in output_cols
        }

        # ---------------------------------------------------------------------
        # INPUT WEIGHTS
        # ---------------------------------------------------------------------

        v = {
            col: LpVariable(
                f"v_{col}_{dmu}",
                lowBound=1e-6
            )
            for col in input_cols
        }

        # ---------------------------------------------------------------------
        # OBJECTIVE
        # ---------------------------------------------------------------------

        prob += lpSum(
            u[r] * dmu_df.loc[dmu, r]
            for r in output_cols
        )

        # ---------------------------------------------------------------------
        # NORMALIZATION
        # ---------------------------------------------------------------------

        prob += (
            lpSum(
                v[i] * dmu_df.loc[dmu, i]
                for i in input_cols
            )
            == 1
        )

        # ---------------------------------------------------------------------
        # CRS CONSTRAINTS
        # ---------------------------------------------------------------------

        for j in dmus:

            prob += (
                lpSum(
                    u[r] * dmu_df.loc[j, r]
                    for r in output_cols
                )
                -
                lpSum(
                    v[i] * dmu_df.loc[j, i]
                    for i in input_cols
                )
                <= 0
            )

        prob.solve(
            PULP_CBC_CMD(msg=False)
        )

        eff = value(prob.objective)

        if eff is None:
            eff = np.nan
        else:
            eff = float(eff)

            # numerical safety
            eff = max(0.0, min(1.0, eff))

        efficiencies.append(eff)

    return efficiencies


# =============================================================================
# JENKS CLASSIFICATION
# =============================================================================

def assign_jenks_with_handling(data, n_classes=5):

    data = pd.Series(data)

    unique_vals = np.unique(data)

    if len(unique_vals) == 1:
        return pd.Series(
            [3] * len(data),
            index=data.index
        )

    if len(unique_vals) < n_classes:
        n_classes = len(unique_vals)

    while n_classes >= 2:

        try:

            breaks = jenkspy.jenks_breaks(
                data.values,
                n_classes=n_classes
            )

            unique_breaks = np.unique(breaks)

            if len(unique_breaks) < len(breaks):
                n_classes -= 1
                continue

            classes = pd.cut(
                data,
                bins=unique_breaks,
                labels=list(
                    range(1, n_classes + 1)
                ),
                include_lowest=True
            )

            return classes.astype(int)

        except Exception:

            n_classes -= 1

    return pd.Series(
        [3] * len(data),
        index=data.index
    )


# =============================================================================
# DISTRICT-MONTH AGGREGATION
# =============================================================================

district_df = (
    df.groupby(
        ["district", "timeperiod"],
        as_index=False
    )
    .agg(
        # ----------------------------------------------------------
        # SENSITIVITY
        # ----------------------------------------------------------

        aged_pop=("sum_aged_population", "sum"),
        young_pop=("sum_young_population", "sum"),
        no_sanitation=("block_nosanitation_hhds_pct", "mean"),
        nco_5_9=("nco_5_9_percent_estimated", "mean"),
        pct_ncd=("pct_ncd", "mean"),

        # ----------------------------------------------------------
        # COPING CAPACITY
        # ----------------------------------------------------------

        health_centers=("HealthCenters", "sum"),
        electricity=("avg_electricity", "mean"),
        piped_water=("block_piped_hhds_pct", "mean")
    )
)

print(
    "District-month observations:",
    len(district_df)
)

# =============================================================================
# MONTHWISE DEA
# =============================================================================

results = []

for month in sorted(
    district_df["timeperiod"].unique()
):

    print(f"Running DEA: {month}")

    month_df = (
        district_df[
            district_df["timeperiod"] == month
        ]
        .copy()
        .reset_index(drop=True)
    )

    # -------------------------------------------------------------------------
    # NORMALIZE
    # -------------------------------------------------------------------------

    scaler = MinMaxScaler()

    scale_cols = [
        "aged_pop",
        "young_pop",
        "no_sanitation",
        "nco_5_9",
        "pct_ncd",
        "health_centers",
        "electricity",
        "piped_water",
    ]

    month_df[scale_cols] = scaler.fit_transform(
        month_df[scale_cols]
    )

    # avoid exact zeros
    month_df[scale_cols] += 1e-6

    # -------------------------------------------------------------------------
    # INVERT COPING CAPACITY
    # -------------------------------------------------------------------------

    month_df["inv_health_centers"] = (
        1 - month_df["health_centers"]
    )

    month_df["inv_electricity"] = (
        1 - month_df["electricity"]
    )

    month_df["inv_piped_water"] = (
        1 - month_df["piped_water"]
    )

    # -------------------------------------------------------------------------
    # CONSTANT OUTPUT
    # -------------------------------------------------------------------------

    month_df["constant_output"] = 1.0

    # -------------------------------------------------------------------------
    # DEA INPUTS
    # -------------------------------------------------------------------------

    INPUTS = [
        "aged_pop",
        "young_pop",
        "no_sanitation",
        "nco_5_9",
        "pct_ncd",
        "inv_health_centers",
        "inv_electricity",
        "inv_piped_water",
    ]

    OUTPUTS = [
        "constant_output"
    ]

    dea_df = month_df.copy()

    dea_df.index = dea_df["district"]

    dea_df["efficiency"] = dea_crs(
        dea_df,
        INPUTS,
        OUTPUTS
    )

    month_df["efficiency"] = (
        dea_df["efficiency"].values
    )

    # -------------------------------------------------------------------------
    # VULNERABILITY
    # -------------------------------------------------------------------------

    month_df["vulnerability_raw"] = (
        1 - month_df["efficiency"]
    )

    # -------------------------------------------------------------------------
    # JENKS NATURAL BREAKS
    # -------------------------------------------------------------------------

    month_df["vulnerability"] = (
        assign_jenks_with_handling(
            month_df["vulnerability_raw"],
            n_classes=5
        )
    )

    results.append(month_df)

# =============================================================================
# COMBINE
# =============================================================================

result_df = pd.concat(
    results,
    ignore_index=True
)

# =============================================================================
# OUTPUT
# =============================================================================

output_cols = [
    "district",
    "timeperiod",

    "aged_pop",
    "young_pop",
    "no_sanitation",
    "nco_5_9",
    "pct_ncd",

    "health_centers",
    "electricity",
    "piped_water",

    "inv_health_centers",
    "inv_electricity",
    "inv_piped_water",

    "efficiency",
    "vulnerability_raw",
    "vulnerability",
]

result_df[output_cols].to_csv(
    OUTPUT_CSV,
    index=False
)

print("\nSaved:", OUTPUT_CSV)

print("\nEfficiency Summary")
print(
    result_df["efficiency"]
    .describe()
)

print("\nVulnerability Class Distribution")
print(
    result_df["vulnerability"]
    .value_counts()
    .sort_index()
)

print("\nMonthly Distribution")

for month in sorted(
    result_df["timeperiod"].unique()
):
    print(f"\n{month}")
    print(
        result_df[
            result_df["timeperiod"] == month
        ]["vulnerability"]
        .value_counts()
        .sort_index()
    )

print("\nPreview")
print(
    result_df.head()
)


# =============================================================================
# APPEND VULNERABILITY TO MASTER_VARIABLES.CSV
# =============================================================================

master = df.copy()

# remove old columns if they exist (prevents _x/_y issues)
for col in ["efficiency", "vulnerability_raw", "vulnerability"]:
    if col in master.columns:
        master = master.drop(columns=[col])

master = master.merge(
    result_df[
        [
            "district",
            "timeperiod",
            "vulnerability",
        ]
    ].drop_duplicates(),
    on=["district", "timeperiod"],
    how="left",
)

master.to_csv(INPUT_CSV, index=False)

print("\nUpdated master file:", INPUT_CSV)

print("\nMissing vulnerability values:")
print(master["vulnerability"].isna().sum())

print("\nPreview:")
print(
    master[
        ["object_id", "district", "timeperiod", "vulnerability"]
    ].head()
)

Input shape: (7222, 26)
District-month observations: 690
Running DEA: 2023_01
Running DEA: 2023_02
Running DEA: 2023_03
Running DEA: 2023_04
Running DEA: 2023_05
Running DEA: 2023_06
Running DEA: 2023_07
Running DEA: 2023_08
Running DEA: 2023_09
Running DEA: 2023_10
Running DEA: 2023_11
Running DEA: 2023_12
Running DEA: 2024_01
Running DEA: 2024_02
Running DEA: 2024_03
Running DEA: 2024_04
Running DEA: 2024_05
Running DEA: 2024_06
Running DEA: 2024_07
Running DEA: 2024_08
Running DEA: 2024_09
Running DEA: 2024_10
Running DEA: 2024_11

Saved: data/vulnerability.csv

Efficiency Summary
count    690.000000
mean       0.981556
std        0.039366
min        0.860161
25%        0.991631
50%        1.000000
75%        1.000000
max        1.000000
Name: efficiency, dtype: float64

Vulnerability Class Distribution
vulnerability
1    552
2     46
3     23
4     23
5     46
Name: count, dtype: int64

Monthly Distribution

2023_01
vulnerability
1    24
2     2
3     1
4     1
5     2
Name: count,